### This analysis is based on the pull aligned continuous bhv variables and neural activity analysis
### The goal of this code is to define GLM model and test the hypothesis that social gaze before pull serves as a evidence accumulation process, and test if the neural profile matches the accumulation hypothesis
### note, the glm will be used to fit the behavioral data only, so it's different from the neuralGLM code

In [ ]:
import numpy as np
import pandas as pd # Added import for Pandas

import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import scipy.stats as st
import scipy.io
from scipy.stats import pearsonr

from sklearn.neighbors import KernelDensity
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score

from dPCA import dPCA

import string
import warnings
import pickle
import json

from scipy.ndimage import gaussian_filter1d

import sys
import os
import glob
import random
from time import time


In [ ]:
# to be able to use the functions in the ana_functions under /3d_recontruction_analysis_self_and_coop_task_neural_analysis/
sys.path.append(os.path.abspath('../3d_recontruction_analysis_self_and_coop_task_neural_analysis/'))

### function - get body part location for each pair of cameras

In [ ]:
from ana_functions.body_part_locs_eachpair import body_part_locs_eachpair
from ana_functions.body_part_locs_singlecam import body_part_locs_singlecam

### function - align the two cameras

In [ ]:
from ana_functions.camera_align import camera_align       

### function - merge the two pairs of cameras

In [ ]:
from ana_functions.camera_merge import camera_merge

### function - find social gaze time point

In [ ]:
from ana_functions.find_socialgaze_timepoint import find_socialgaze_timepoint
from ana_functions.find_socialgaze_timepoint_singlecam import find_socialgaze_timepoint_singlecam
from ana_functions.find_socialgaze_timepoint_singlecam_wholebody import find_socialgaze_timepoint_singlecam_wholebody
from ana_functions.find_socialgaze_timepoint_singlecam_wholebody_2 import find_socialgaze_timepoint_singlecam_wholebody_2


### function - define time point of behavioral events

In [ ]:
from ana_functions.bhv_events_timepoint import bhv_events_timepoint
from ana_functions.bhv_events_timepoint_singlecam import bhv_events_timepoint_singlecam

### function - plot behavioral events

In [ ]:
from ana_functions.plot_bhv_events import plot_bhv_events
from ana_functions.plot_bhv_events_levertube import plot_bhv_events_levertube
from ana_functions.plot_continuous_bhv_var_singlecam import plot_continuous_bhv_var_singlecam
from ana_functions.draw_self_loop import draw_self_loop
import matplotlib.patches as mpatches 
from matplotlib.collections import PatchCollection

### function - plot inter-pull interval

In [ ]:
from ana_functions.plot_interpull_interval import plot_interpull_interval

### function - make demo videos with skeleton and inportant vectors

In [ ]:
from ana_functions.tracking_video_singlecam_demo import tracking_video_singlecam_demo
from ana_functions.tracking_video_singlecam_wholebody_demo import tracking_video_singlecam_wholebody_demo
from ana_functions.tracking_video_singlecam_wholebody_withNeuron_demo import tracking_video_singlecam_wholebody_withNeuron_demo
from ana_functions.tracking_video_singlecam_wholebody_withNeuron_sepbhv_demo import tracking_video_singlecam_wholebody_withNeuron_sepbhv_demo
from ana_functions.tracking_frame_singlecam_wholebody_withNeuron_sepbhv_demo import tracking_frame_singlecam_wholebody_withNeuron_sepbhv_demo

### function - interval between all behavioral events

In [ ]:
from ana_functions.bhv_events_interval import bhv_events_interval

### function - spike analysis

In [ ]:
from ana_functions.spike_analysis_FR_calculation import spike_analysis_FR_calculation
from ana_functions.plot_spike_triggered_singlecam_bhvevent import plot_spike_triggered_singlecam_bhvevent
from ana_functions.plot_bhv_events_aligned_FR import plot_bhv_events_aligned_FR
from ana_functions.plot_strategy_aligned_FR import plot_strategy_aligned_FR

### function - PCA projection

In [ ]:
from ana_functions.PCA_around_bhv_events import PCA_around_bhv_events
from ana_functions.PCA_around_bhv_events_video import PCA_around_bhv_events_video
from ana_functions.confidence_ellipse import confidence_ellipse

### function - other useful functions

In [ ]:
# for defining the meaningful social gaze (the continuous gaze distribution that is closest to the pull) 
from ana_functions.keep_closest_cluster_single_trial import keep_closest_cluster_single_trial

In [ ]:
# get more information for each pull: number of preceding failed pull and time since last reward/successful pull
from ana_functions.get_pull_infos import get_pull_infos


In [ ]:
from functions.continuous_variable_glm import continuous_variable_glm
from functions.continuous_variable_glm_shortlist_prediction import continuous_variable_glm_shortlist_prediction
from functions.continuous_variable_create_data_forGLM_old import continuous_variable_create_data_forGLM

## Analyze each session

### prepare the basic behavioral data (especially the time stamps for each bhv events)

In [ ]:
# instead of using gaze angle threshold, use the target rectagon to deside gaze info
# ...need to update
sqr_thres_tubelever = 75 # draw the square around tube and lever
sqr_thres_face = 1.15 # a ratio for defining face boundary
sqr_thres_body = 4 # how many times to enlongate the face box boundry to the body


# get the fps of the analyzed video
fps = 30

# get the fs for neural recording
fs_spikes = 20000
fs_lfp = 1000

# frame number of the demo video
nframes = 0.5*30 # second*30fps
# nframes = 45*30 # second*30fps

# re-analyze the video or not
reanalyze_video = 0
redo_anystep = 0

# do OFC sessions or DLPFC sessions
do_OFC = 0
do_DLPFC  = 1
if do_OFC:
    savefile_sufix = '_OFCs'
elif do_DLPFC:
    savefile_sufix = '_DLPFCs'
else:
    savefile_sufix = ''
    
# all the videos (no misaligned ones)
# aligned with the audio
# get the session start time from "videosound_bhv_sync.py/.ipynb"
# currently the session_start_time will be manually typed in. It can be updated after a better method is used


# dodson ginger for dlpfc (dmpfc)
# dodson selene for ofc
if 1:
    if do_DLPFC:
        neural_record_conditions = [
                        '20240531_Dodson_MC', '20240603_Dodson_MC_and_SR', '20240603_Dodson_MC_and_SR',
                        '20240604_Dodson_MC', '20240605_Dodson_MC_and_SR', '20240605_Dodson_MC_and_SR',

                        '20240606_Dodson_MC_and_SR', '20240606_Dodson_MC_and_SR', '20240607_Dodson_SR',
                        '20240610_Dodson_MC', '20240611_Dodson_SR', '20240612_Dodson_MC',

                        '20240613_Dodson_SR', '20240620_Dodson_SR', '20240719_Dodson_MC',
                        '20250129_Dodson_MC', '20250130_Dodson_SR', '20250131_Dodson_MC',

                        '20250210_Dodson_SR_withKoala', '20250211_Dodson_MC_withKoala',
                        '20250212_Dodson_SR_withKoala', '20250214_Dodson_MC_withKoala',
                        '20250217_Dodson_SR_withKoala', '20250218_Dodson_MC_withKoala',

                        '20250219_Dodson_SR_withKoala', '20250220_Dodson_MC_withKoala',
                        '20250224_Dodson_KoalaAL_withKoala', '20250226_Dodson_MC_withKoala',
                        '20250227_Dodson_KoalaAL_withKoala', '20250228_Dodson_DodsonAL_withKoala',

                        '20250304_Dodson_DodsonAL_withKoala', '20250305_Dodson_MC_withKoala',
                        '20250306_Dodson_KoalaAL_withKoala', '20250307_Dodson_DodsonAL_withKoala',
                        '20250310_Dodson_MC_withKoala', '20250312_Dodson_NV_withKoala',

                        '20250313_Dodson_NV_withKoala', '20250314_Dodson_NV_withKoala',
                        '20250401_Dodson_MC_withKanga', '20250402_Dodson_MC_withKanga',
                        '20250403_Dodson_MC_withKanga', '20250404_Dodson_SR_withKanga',

                        '20250407_Dodson_SR_withKanga', '20250408_Dodson_SR_withKanga',
                        '20250409_Dodson_MC_withKanga', '20250415_Dodson_MC_withKanga',
                        # '20250416_Dodson_SR_withKanga',
                        '20250417_Dodson_MC_withKanga',

                        '20250418_Dodson_SR_withKanga', '20250421_Dodson_SR_withKanga',
                        '20250422_Dodson_MC_withKanga', '20250422_Dodson_SR_withKanga',
                        '20250423_Dodson_MC_withKanga', '20250423_Dodson_SR_withKanga',

                        '20250424_Dodson_NV_withKanga', '20250424_Dodson_MC_withKanga',
                        '20250424_Dodson_SR_withKanga', '20250425_Dodson_NV_withKanga',
                        '20250425_Dodson_SR_withKanga', '20250428_Dodson_NV_withKanga',

                        '20250428_Dodson_MC_withKanga', '20250428_Dodson_SR_withKanga',
                        '20250429_Dodson_NV_withKanga', '20250429_Dodson_MC_withKanga',
                        '20250429_Dodson_SR_withKanga', '20250430_Dodson_NV_withKanga',

                        '20250430_Dodson_MC_withKanga', '20250430_Dodson_SR_withKanga',
                    ]
        task_conditions = [
                        'MC', 'MC', 'SR', 'MC', 'MC', 'SR',
                        'MC', 'SR', 'SR', 'MC', 'SR', 'MC',
                        'SR', 'SR', 'MC', 'MC_withGingerNew', 'SR_withGingerNew', 'MC_withGingerNew',

                        'SR_withKoala', 'MC_withKoala', 'SR_withKoala',
                        'MC_withKoala', 'SR_withKoala', 'MC_withKoala',

                        'SR_withKoala', 'MC_withKoala', 'MC_KoalaAuto_withKoala',
                        'MC_withKoala', 'MC_KoalaAuto_withKoala', 'MC_DodsonAuto_withKoala',

                        'MC_DodsonAuto_withKoala', 'MC_withKoala', 'MC_KoalaAuto_withKoala',
                        'MC_DodsonAuto_withKoala', 'MC_withKoala', 'NV_withKoala',

                        'NV_withKoala', 'NV_withKoala', 'MC_withKanga',
                        'MC_withKanga', 'MC_withKanga', 'SR_withKanga',

                        'SR_withKanga', 'SR_withKanga', 'MC_withKanga',
                        'MC_withKanga',
                        # 'SR_withKanga',
                        'MC_withKanga', 'SR_withKanga', 'SR_withKanga', 'MC_withKanga', 'SR_withKanga',

                        'MC_withKanga', 'SR_withKanga', 'NV_withKanga',
                        'MC_withKanga', 'SR_withKanga', 'NV_withKanga',

                        'SR_withKanga', 'NV_withKanga', 'MC_withKanga',
                        'SR_withKanga', 'NV_withKanga', 'MC_withKanga',

                        'SR_withKanga', 'NV_withKanga', 'MC_withKanga', 'SR_withKanga',
                    ]
        dates_list = [
                        '20240531', '20240603_MC', '20240603_SR', '20240604', '20240605_MC', '20240605_SR',
                        '20240606_MC', '20240606_SR', '20240607', '20240610_MC', '20240611', '20240612',

                        '20240613', '20240620', '20240719',
                        '20250129', '20250130', '20250131',

                        '20250210', '20250211', '20250212', '20250214', '20250217', '20250218',
                        '20250219', '20250220', '20250224', '20250226', '20250227', '20250228',

                        '20250304', '20250305', '20250306', '20250307', '20250310', '20250312',
                        '20250313', '20250314',

                        '20250401', '20250402', '20250403', '20250404', '20250407', '20250408',
                        '20250409',

                        '20250415',
                        # '20250416',
                        '20250417', '20250418', '20250421', '20250422', '20250422_SR',

                        '20250423', '20250423_SR', '20250424', '20250424_MC', '20250424_SR',
                        '20250425', '20250425_SR',

                        '20250428_NV', '20250428_MC', '20250428_SR',
                        '20250429_NV', '20250429_MC', '20250429_SR',

                        '20250430_NV', '20250430_MC', '20250430_SR',
                    ]
        videodates_list = [
                        '20240531', '20240603', '20240603', '20240604', '20240605', '20240605',
                        '20240606', '20240606', '20240607', '20240610_MC', '20240611', '20240612',

                        '20240613', '20240620', '20240719',
                        '20250129', '20250130', '20250131',

                        '20250210', '20250211', '20250212', '20250214', '20250217', '20250218',
                        '20250219', '20250220', '20250224', '20250226', '20250227', '20250228',

                        '20250304', '20250305', '20250306', '20250307', '20250310', '20250312',
                        '20250313', '20250314',

                        '20250401', '20250402', '20250403', '20250404', '20250407', '20250408',
                        '20250409',

                        '20250415',
                        # '20250416',
                        '20250417', '20250418', '20250421', '20250422', '20250422_SR',

                        '20250423', '20250423_SR', '20250424', '20250424_MC', '20250424_SR',
                        '20250425', '20250425_SR',

                        '20250428_NV', '20250428_MC', '20250428_SR',
                        '20250429_NV', '20250429_MC', '20250429_SR',

                        '20250430_NV', '20250430_MC', '20250430_SR',
                    ] # to deal with the sessions that MC and SR were in the same session
        session_start_times = [
                        0.00, 340, 340, 72.0, 60.1, 60.1,
                        82.2, 82.2, 35.8, 0.00, 29.2, 35.8,

                        62.5, 71.5, 54.4,
                        0.00, 0.00, 0.00,

                        0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
                        0.00, 0.00, 0.00, 0.00, 0.00, 0.00,

                        0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
                        0.00, 0.00,

                        0.00, 0.00, 73.5, 0.00, 76.1, 81.5,
                        0.00,

                        363,
                        # 0.00,
                        79.0, 162.6, 231.9, 109, 0.00,

                        0.00, 0.00, 0.00, 0.00, 0.00,
                        0.00, 93.0,

                        0.00, 0.00, 0.00,
                        0.00, 0.00, 0.00,

                        0.00, 274.4, 0.00,
                    ]
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = [ 'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0', #'Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             
                              ]
        animal1_fixedorders = ['dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',# 'dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson',
                              ]
        recordedanimals = animal1_fixedorders 
        animal2_fixedorders = ['ginger','ginger','ginger','ginger','ginger','ginger','ginger','ginger','ginger',
                               'ginger','ginger','ginger','ginger','ginger','ginger','gingerNew','gingerNew','gingerNew',
                               'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala',
                               'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala',
                               'koala', 'koala', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', # 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                              ]

        animal1_filenames = ["Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",# "Dodson",
                             'Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson',
                             'Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson',
                             'Dodson','Dodson','Dodson','Dodson','Dodson',
                            ]
        animal2_filenames = ["Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger",
                             "Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger",
                             "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala",
                             "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala",
                             "Koala", "Koala", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga", # "Kanga",
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                            ]
        
    elif do_OFC:
        neural_record_conditions = [
                        '20260219_Dodson_OFC_33turns_SRwithSelene',  '20260303_Dodson_OFC_33turns_1sMCwithSelene',
                        '20260303_Dodson_OFC_33turns_SRwithSelene',  '20260304_Dodson_OFC_33turns_1sMCwithSelene',
                        '20260304_Dodson_OFC_33turns_SRwithSelene',  '20260305_Dodson_OFC_33turns_1sMCwithSelene',
                        '20260309_Dodson_OFC_33turns_1sMCwithKanga', '20260309_Dodson_OFC_33turns_SRwithKanga',
                        '20260310_Dodson_OFC_33turns_1sMCwithKanga', '20260310_Dodson_OFC_33turns_SRwithKanga',
                        '20260311_Dodson_OFC_33turns_1sMCwithKanga', '20260311_Dodson_OFC_33turns_SRwithKanga',
                        '20260312_Dodson_OFC_33turns_1sMCwithKanga', '20260312_Dodson_OFC_33turns_SRwithKanga',
                        '20260313_Dodson_OFC_33turns_1sMCwithKanga', '20260313_Dodson_OFC_33turns_SRwithKanga',
            
                        '20260317_Dodson_OFC_33turns_1sMCwithKanga', '20260317_Dodson_OFC_33turns_SRwithKanga',
                        '20260318_Dodson_OFC_33turns_1sMCwithKanga', # '20260318_Dodson_OFC_33turns_SRwithKanga',
                        '20260319_Dodson_OFC_33turns_1sMCwithKanga', '20260319_Dodson_OFC_33turns_SRwithKanga',
                        '20260323_Dodson_OFC_32turns_1sMCwithKanga', '20260323_Dodson_OFC_32turns_SRwithKanga',
                        '20260324_Dodson_OFC_32turns_1sMCwithKanga', '20260324_Dodson_OFC_32turns_SRwithKanga',
                    ]
        task_conditions = [
                        'SR', 'MC', 'SR', 'MC', 'SR', 'MC', 
                        'MC', 'SR', 'MC', 'SR', 'MC', 'SR',
                        'MC', 'SR', 'MC', 'SR', 'MC', 'SR',
                        'MC',       'MC', 'SR', 'MC', 'SR',
                        'MC', 'SR', 
                    ]
        dates_list = [
                        '20260219', '20260303',    '20260303_SR', '20260304',    '20260304_SR', '20260305', 
                        '20260309', '20260309_SR', '20260310',    '20260310_SR', '20260311',    '20260311_SR',
                        '20260312', '20260312_SR', '20260313',    '20260313_SR', '20260317',    '20260317_SR',
                        '20260318',                '20260319',    '20260319_SR', '20260323',    '20260323_SR',
                        '20260324', '20260324_SR',
                    ]
        videodates_list = [
                        '20260219', '20260303',    '20260303_SR', '20260304',    '20260304_SR', '20260305', 
                        '20260309', '20260309_SR', '20260310',    '20260310_SR', '20260311',    '20260311_SR',
                        '20260312', '20260312_SR', '20260313',    '20260313_SR', '20260317',    '20260317_SR',
                        '20260318',                '20260319',    '20260319_SR', '20260323',    '20260323_SR',
                        '20260324', '20260324_SR',
                    ] 
        
        session_start_times = [
                        188.7, 0.00, 0.00, 0.00, 0.00,  0.00, 
                         0.00, 0.00, 0.00, 0.00, 0.00,  0.00, 
                         0.00, 0.00, 0.00, 0.00, 0.00,  0.00, 
                         0.00,       0.00, 0.00, 0.00, 129.5,
                        116.2, 0.00,
                    ]
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = [ 'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              'Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              'Dev1/ai9',           'Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              'Dev1/ai9','Dev1/ai9',
                              ]
        animal1_fixedorders = ['dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson',         'dodson','dodson','dodson','dodson',
                               'dodson','dodson',
                              ]
        recordedanimals = animal1_fixedorders 
        animal2_fixedorders = ['selene','selene','selene','selene','selene','selene',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga',          'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga',
                              ]

        animal1_filenames = ["Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson",         "Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson",
                            ]
        animal2_filenames = ['Selene','Selene','Selene','Selene','Selene','Selene',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga',          'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga',
                            ]


    
# dannon kanga
if 1:
    if do_DLPFC:
        neural_record_conditions = [
                        '20240508_Kanga_SR', '20240509_Kanga_MC', '20240513_Kanga_MC',
                        '20240514_Kanga_SR', '20240523_Kanga_MC', '20240524_Kanga_SR',

                        '20240606_Kanga_MC', '20240613_Kanga_MC_DannonAuto',
                        '20240614_Kanga_MC_DannonAuto', '20240617_Kanga_MC_DannonAuto',
                        '20240618_Kanga_MC_KangaAuto', '20240619_Kanga_MC_KangaAuto',

                        '20240620_Kanga_MC_KangaAuto', '20240621_1_Kanga_NoVis',
                        '20240624_Kanga_NoVis', '20240626_Kanga_NoVis',

                        '20240808_Kanga_MC_withGinger', '20240809_Kanga_MC_withGinger',
                        '20240812_Kanga_MC_withGinger', '20240813_Kanga_MC_withKoala',
                        '20240814_Kanga_MC_withKoala', '20240815_Kanga_MC_withKoala',

                        '20240819_Kanga_MC_withVermelho', '20240821_Kanga_MC_withVermelho',
                        '20240822_Kanga_MC_withVermelho',

                        '20250415_Kanga_MC_withDodson', '20250416_Kanga_SR_withDodson',
                        '20250417_Kanga_MC_withDodson', '20250418_Kanga_SR_withDodson',
                        '20250421_Kanga_SR_withDodson', '20250422_Kanga_MC_withDodson',

                        '20250422_Kanga_SR_withDodson', '20250423_Kanga_MC_withDodson',
                        '20250423_Kanga_SR_withDodson',

                        '20250424_Kanga_NV_withDodson', '20250424_Kanga_MC_withDodson',
                        '20250424_Kanga_SR_withDodson', '20250425_Kanga_NV_withDodson',
                        '20250425_Kanga_SR_withDodson',

                        '20250428_Kanga_NV_withDodson', '20250428_Kanga_MC_withDodson',
                        '20250428_Kanga_SR_withDodson', '20250429_Kanga_NV_withDodson',
                        '20250429_Kanga_MC_withDodson', '20250429_Kanga_SR_withDodson',

                        '20250430_Kanga_NV_withDodson', '20250430_Kanga_MC_withDodson',
                        '20250430_Kanga_SR_withDodson',
                    ]
        dates_list = [
                        "20240508", "20240509", "20240513", "20240514", "20240523", "20240524",
                        "20240606", "20240613", "20240614", "20240617", "20240618", "20240619",
                        "20240620", "20240621_1", "20240624", "20240626",

                        "20240808", "20240809", "20240812", "20240813", "20240814", "20240815",
                        "20240819", "20240821", "20240822",

                        "20250415", "20250416", "20250417", "20250418", "20250421", "20250422",
                        "20250422_SR",

                        '20250423', '20250423_SR', '20250424', '20250424_MC', '20250424_SR',
                        '20250425', '20250425_SR',

                        '20250428_NV', '20250428_MC', '20250428_SR',
                        '20250429_NV', '20250429_MC', '20250429_SR',

                        '20250430_NV', '20250430_MC', '20250430_SR',
                    ]
        videodates_list = dates_list
        task_conditions = [
                        'SR', 'MC', 'MC', 'SR', 'MC', 'SR',
                        'MC', 'MC_DannonAuto', 'MC_DannonAuto', 'MC_DannonAuto',
                        'MC_KangaAuto', 'MC_KangaAuto',

                        'MC_KangaAuto', 'NV', 'NV', 'NV',

                        'MC_withGinger', 'MC_withGinger', 'MC_withGinger',
                        'MC_withKoala', 'MC_withKoala', 'MC_withKoala',

                        'MC_withVermelho', 'MC_withVermelho', 'MC_withVermelho',

                        'MC_withDodson', 'SR_withDodson', 'MC_withDodson',
                        'SR_withDodson', 'SR_withDodson', 'MC_withDodson',

                        'SR_withDodson', 'MC_withDodson', 'SR_withDodson',

                        'NV_withDodson', 'MC_withDodson', 'SR_withDodson',
                        'NV_withDodson', 'SR_withDodson',

                        'NV_withDodson', 'MC_withDodson', 'SR_withDodson',
                        'NV_withDodson', 'MC_withDodson', 'SR_withDodson',

                        'NV_withDodson', 'MC_withDodson', 'SR_withDodson',
                    ]
        session_start_times = [
                        0.00, 36.0, 69.5, 0.00, 62.0, 0.00,
                        89.0, 0.00, 0.00, 0.00, 165.8, 96.0,
            
                        0.00, 0.00, 0.00, 48.0,
                        59.2, 49.5, 40.0, 50.0, 0.00, 69.8,
            
                        85.0, 212.9, 68.5,
                        363, 0.00, 79.0, 162.6, 231.9, 109,
            
                        0.00,
                        0.00, 0.00, 0.00, 0.00, 0.00,

                        0.00, 93.0,

                        0.00, 0.00, 0.00, 0.00, 0.00,
                        0.00,

                        0.00, 274.4, 0.00,
                    ]
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = ['Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai0','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                             'Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              ]
        
        animal1_fixedorders = ['dannon','dannon','dannon','dannon','dannon','dannon','dannon','dannon',
                               'dannon','dannon','dannon','dannon','dannon','dannon','dannon','dannon',
                               'ginger','ginger','ginger','koala','koala','koala','vermelho','vermelho',
                               'vermelho','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                              ]
        animal2_fixedorders = ['kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                              ]
        recordedanimals = animal2_fixedorders

        animal1_filenames = ["Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon",
                             "Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon",
                             "Ginger","Ginger","Ginger", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga",
                              "Kanga","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             
                            ]
        animal2_filenames = ["Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Koala","Koala","Koala","Vermelho","Vermelho",
                             "Vermelho","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                            ]
        
    elif do_OFC:
        neural_record_conditions = [
                        '20260309_Kanga_OFC_31turns_1sMCwithDodson', '20260309_Kanga_OFC_31turns_SRwithDodson',
                        '20260310_Kanga_OFC_31turns_1sMCwithDodson', '20260310_Kanga_OFC_31turns_SRwithDodson',
                        '20260311_Kanga_OFC_31turns_1sMCwithDodson', '20260311_Kanga_OFC_31turns_SRwithDodson',
                        '20260312_Kanga_OFC_31turns_1sMCwithDodson', '20260312_Kanga_OFC_31turns_SRwithDodson',
                        '20260313_Kanga_OFC_31turns_1sMCwithDodson', '20260313_Kanga_OFC_31turns_SRwithDodson',
                        '20260317_Kanga_OFC_31turns_1sMCwithDodson', '20260317_Kanga_OFC_31turns_SRwithDodson',
                        '20260318_Kanga_OFC_31turns_1sMCwithDodson', '20260318_Kanga_OFC_31turns_SRwithDodson',
                        '20260319_Kanga_OFC_31turns_1sMCwithDodson', '20260319_Kanga_OFC_31turns_SRwithDodson',
                        '20260323_Kanga_OFC_31turns_1sMCwithDodson', '20260323_Kanga_OFC_31turns_SRwithDodson',
                        '20260324_Kanga_OFC_31turns_1sMCwithDodson', '20260324_Kanga_OFC_31turns_SRwithDodson',
                        '20260326_Kanga_OFC_31turns_1sMCwithDodson', '20260326_Kanga_OFC_31turns_SRwithDodson',
                        '20260330_Kanga_OFC_30dot5turns_SRwithDodson',   '20260331_Kanga_OFC_30dot5turns_1sMCwithDodson',
                        '20260403_Kanga_OFC_30dot5turns_1sMCwithDodson', '20260406_Kanga_OFC_30dot5turns_1sMCwithDodson',
                        '20260406_Kanga_OFC_30dot5turns_SRwithDodson',
            
                        # dannon kanga
                        '20260409_Kanga_OFC_30dot5turns_1sMCwithDannon', '20260410_Kanga_OFC_30dot5turns_1sMCwithDannon',
                        '20260410_Kanga_OFC_30dot5turns_SRwithDannon',   '20260413_Kanga_OFC_30dot5turns_1sMCwithDannon',
                        '20260421_Kanga_OFC_30dot5turns_1sMCwithDannon',
                    ]
        task_conditions = [
                        'MC', 'SR', 'MC', 'SR', 'MC', 'SR',
                        'MC', 'SR', 'MC', 'SR', 'MC', 'SR',
                        'MC', 'SR', 'MC', 'SR', 'MC', 'SR',
                        'MC', 'SR', 'MC', 'SR', 'SR', 'MC',
                        'MC', 'MC', 'SR',
            
                        'MC', 'MC', 'SR', 'MC', 'MC',
                    ]
        dates_list = [
                        '20260309', '20260309_SR', '20260310', '20260310_SR', '20260311',    '20260311_SR',
                        '20260312', '20260312_SR', '20260313', '20260313_SR', '20260317',    '20260317_SR',
                        '20260318', '20260318_SR', '20260319', '20260319_SR', '20260323',    '20260323_SR',
                        '20260324', '20260324_SR', '20260326', '20260326_SR', '20260330_SR', '20260331',
                        '20260403', '20260406', '20260406_SR',
                    
                        '20260409', '20260410', '20260410_SR', '20260413',    '20260421',
                    ]
        videodates_list = [
                        '20260309', '20260309_SR', '20260310', '20260310_SR', '20260311',    '20260311_SR',
                        '20260312', '20260312_SR', '20260313', '20260313_SR', '20260317',    '20260317_SR',
                        '20260318', '20260318_SR', '20260319', '20260319_SR', '20260323',    '20260323_SR',
                        '20260324', '20260324_SR', '20260326', '20260326_SR', '20260330_SR', '20260331',
                        '20260403', '20260406', '20260406_SR', 
            
                        '20260409', '20260410', '20260410_SR', '20260413',    '20260421',
                    ] 
        
        session_start_times = [
                         0.00, 0.00, 0.00, 0.00, 0.00,  0.00, 
                         0.00, 0.00, 0.00, 0.00, 0.00,  0.00, 
                         0.00, 0.00, 0.00, 0.00, 0.00, 129.5,
                        116.2, 0.00, 0.00, 0.00, 0.00, 49.50,
                         0.00, 0.00, 0.00,
            
                         0.00, 0.00, 0.00, 0.00, 0.00,
                    ]
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = [ 'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0', 
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0', 
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0', 
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              ]
        animal1_fixedorders = [
                               'dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson', 
                               'dodson','dodson','dodson',
             
                               'dannon','dannon','dannon','dannon','dannon',
                              ]
        animal2_fixedorders = [
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 
                               'kanga', 'kanga', 'kanga', 
            
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                              ]
        recordedanimals = animal2_fixedorders 

        animal1_filenames = [
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson",
            
                             "Dannon","Dannon","Dannon","Dannon","Dannon",
                            ]
        animal2_filenames = [
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 
                             'Kanga', 'Kanga', 'Kanga', 
            
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 
                            ]
    

    
# a test case
if 0:
    if do_DLPFC:
        if 0: # kanga example
            neural_record_conditions = ['20240606_Kanga_MC']
            dates_list = ["20240606"]
            videodates_list = dates_list
            task_conditions = ['MC']
            session_start_times = [89] # in second
            kilosortvers = [4]
            trig_channelnames = ['Dev1/ai0']
            animal1_fixedorders = ['dannon']
            animal2_fixedorders = ['kanga']
            recordedanimals = animal2_fixedorders
            animal1_filenames = ["Dannon"]
            animal2_filenames = ["Kanga"]
        if 0: # dodson example 
            neural_record_conditions = ['20250415_Dodson_MC_withKanga']
            dates_list = ["20250415"]
            videodates_list = dates_list
            task_conditions = ['MC_withKanga']
            session_start_times = [363] # in second
            kilosortvers = [4]
            trig_channelnames = ['Dev1/ai0']
            animal1_fixedorders = ['dodson']
            recordedanimals = animal1_fixedorders
            animal2_fixedorders = ['kanga']
            animal1_filenames = ["Dodson"]
            animal2_filenames = ["Kanga"]
    #
    elif do_OFC:
        if 1: # kanga example
            neural_record_conditions = [ '20260309_Kanga_OFC_31turns_1sMCwithDodson',]
            task_conditions = [ 'MC', ]
            dates_list = [ '20260309', ]
            videodates_list = [ '20260309', ] 
            session_start_times = [0.00, ]
            kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
            trig_channelnames = [ 'Dev1/ai0',]
            animal1_fixedorders = ['dodson',]
            animal2_fixedorders = [ 'kanga',]
            recordedanimals = animal2_fixedorders 
            animal1_filenames = [ "Dodson",]
            animal2_filenames = ['Kanga', ]
        if 0: # dodson example
            neural_record_conditions = [ '20260309_Dodson_OFC_33turns_1sMCwithKanga',]
            task_conditions = [ 'MC', ]
            dates_list = [ '20260309', ]
            videodates_list = [ '20260309', ] 
            session_start_times = [0.00, ]
            kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
            trig_channelnames = [ 'Dev1/ai9',]
            animal1_fixedorders = ['dodson',]
            animal2_fixedorders = [ 'kanga',]
            recordedanimals = animal1_fixedorders 
            animal1_filenames = [ "Dodson",]
            animal2_filenames = ['Kanga', ]
            
    
ndates = np.shape(dates_list)[0]

session_start_frames = session_start_times * fps # fps is 30Hz

# totalsess_time = 600

# video tracking results info
animalnames_videotrack = ['dodson','scorch'] # does not really mean dodson and scorch, instead, indicate animal1 and animal2
bodypartnames_videotrack = ['rightTuft','whiteBlaze','leftTuft','rightEye','leftEye','mouth']


# which camera to analyzed
cameraID = 'camera-2'
cameraID_short = 'cam2'

considerlevertube = 1
considertubeonly = 0

# location of levers and tubes for camera 2
# # camera 1
# lever_locs_camI = {'dodson':np.array([645,600]),'scorch':np.array([425,435])}
# tube_locs_camI  = {'dodson':np.array([1350,630]),'scorch':np.array([555,345])}
# # camera 2
# # location of the estimiated middle of the box
lever_locs_camI = {'dodson':np.array([1325,615]),'scorch':np.array([560,615])}
# # location of the estimated lever
# lever_locs_camI = {'dodson':np.array([1335,715]),'scorch':np.array([550,715])}
tube_locs_camI  = {'dodson':np.array([1550,515]),'scorch':np.array([350,515])}
# # old
# # lever_locs_camI = {'dodson':np.array([1335,715]),'scorch':np.array([550,715])}
# # tube_locs_camI  = {'dodson':np.array([1650,490]),'scorch':np.array([250,490])}
# # camera 3
# lever_locs_camI = {'dodson':np.array([1580,440]),'scorch':np.array([1296,540])}
# tube_locs_camI  = {'dodson':np.array([1470,375]),'scorch':np.array([805,475])}


if np.shape(session_start_times)[0] != np.shape(dates_list)[0]:
    exit()

    
# define glm data summarizing data set    
glm_datas_all_dates = dict.fromkeys(dates_list, [])

glm_datas_shortlist_prediction_all_dates = dict.fromkeys(dates_list, [])

pre_data_for_GLM_alldates = dict.fromkeys(dates_list, [])

# where to save the summarizing data
data_saved_folder = '/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/3d_recontruction_analysis_self_and_coop_task_data_saved/'

# neural data folder
neural_data_folder = '/gpfs/radev/pi/nandy/jadi_gibbs_data/Marmoset_neural_recording/'

    

In [ ]:
print(np.shape(neural_record_conditions))
print(np.shape(task_conditions))
print(np.shape(dates_list))
print(np.shape(videodates_list)) 
print(np.shape(session_start_times))

print(np.shape(kilosortvers))

print(np.shape(trig_channelnames))
print(np.shape(animal1_fixedorders)) 
print(np.shape(recordedanimals))
print(np.shape(animal2_fixedorders))

print(np.shape(animal1_filenames))
print(np.shape(animal2_filenames))  

In [ ]:
# basic behavior analysis (define time stamps for each bhv events, etc)

try:
    if redo_anystep:
        dummy
    
    # dummy 
    
    #
    print('loading all data')
    
    # load saved data
    data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_glm'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
    
    with open(data_saved_subfolder+'/glm_datas_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        glm_datas_all_dates = pickle.load(f)
   
    with open(data_saved_subfolder+'/glm_datas_shortlist_prediction_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        glm_datas_shortlist_prediction_all_dates = pickle.load(f)
    
    with open(data_saved_subfolder+'/pre_data_for_GLM_alldates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        pre_data_for_GLM_alldates = pickle.load(f)
        
        
    print('all data from all dates are loaded')

except:

    print('analyze all dates')

    for idate in np.arange(0,ndates,1):
    
        date_tgt = dates_list[idate]
        videodate_tgt = videodates_list[idate]
        
        neural_record_condition = neural_record_conditions[idate]
        
        session_start_time = session_start_times[idate]
        
        kilosortver = kilosortvers[idate]

        trig_channelname = trig_channelnames[idate]
        
        animal1_filename = animal1_filenames[idate]
        animal2_filename = animal2_filenames[idate]
        
        animal1_fixedorder = [animal1_fixedorders[idate]]
        animal2_fixedorder = [animal2_fixedorders[idate]]
        
        recordedanimal = recordedanimals[idate]
        
        # load behavioral results
        if not do_OFC:
            try:
                bhv_data_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/marmoset_tracking_bhv_data_cooperation_task_DLPFCs/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
                trial_record_json = glob.glob(bhv_data_path +date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_TrialRecord_" + "*.json")
                bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_bhv_data_" + "*.json")
                session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_session_info_" + "*.json")
                ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_ni_data_" + "*.json")
                #
                trial_record = pd.read_json(trial_record_json[0])
                bhv_data = pd.read_json(bhv_data_json[0])
                session_info = pd.read_json(session_info_json[0])
                # 
                with open(ni_data_json[0]) as f:
                    for line in f:
                        ni_data=json.loads(line)   
            except:
                bhv_data_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/marmoset_tracking_bhv_data_cooperation_task_DLPFCs/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
                trial_record_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_TrialRecord_" + "*.json")
                bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_bhv_data_" + "*.json")
                session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_session_info_" + "*.json")
                ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_ni_data_" + "*.json")
                #
                trial_record = pd.read_json(trial_record_json[0])
                bhv_data = pd.read_json(bhv_data_json[0])
                session_info = pd.read_json(session_info_json[0])
                #
                with open(ni_data_json[0]) as f:
                    for line in f:
                        ni_data=json.loads(line)
        
        elif do_OFC:
            try:
                bhv_data_path = "/gpfs/marilyn/pi/nandy/VideoTracker_SocialInter/marmoset_tracking_bhv_data_cooperation_task_OFCs/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
                trial_record_json = glob.glob(bhv_data_path +date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_TrialRecord_" + "*.json")
                bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_bhv_data_" + "*.json")
                session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_session_info_" + "*.json")
                ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_ni_data_" + "*.json")
                #
                trial_record = pd.read_json(trial_record_json[0])
                bhv_data = pd.read_json(bhv_data_json[0])
                session_info = pd.read_json(session_info_json[0])
                # 
                with open(ni_data_json[0]) as f:
                    for line in f:
                        ni_data=json.loads(line)   
            except:
                bhv_data_path = "/gpfs/marilyn/pi/nandy/VideoTracker_SocialInter/marmoset_tracking_bhv_data_cooperation_task_OFCs/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
                trial_record_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_TrialRecord_" + "*.json")
                bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_bhv_data_" + "*.json")
                session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_session_info_" + "*.json")
                ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_ni_data_" + "*.json")
                #
                trial_record = pd.read_json(trial_record_json[0])
                bhv_data = pd.read_json(bhv_data_json[0])
                session_info = pd.read_json(session_info_json[0])
                #
                with open(ni_data_json[0]) as f:
                    for line in f:
                        ni_data=json.loads(line)

        # get animal info from the session information
        animal1 = session_info['lever1_animal'][0].lower()
        animal2 = session_info['lever2_animal'][0].lower()

        
        # get task type and cooperation threshold
        try:
            coop_thres = session_info["pulltime_thres"][0]
            tasktype = session_info["task_type"][0]
        except:
            coop_thres = 0
            tasktype = 1
    
            
        # clean up the trial_record
        warnings.filterwarnings('ignore')
        trial_record_clean = pd.DataFrame(columns=trial_record.columns)
        # for itrial in np.arange(0,np.max(trial_record['trial_number']),1):
        for itrial in trial_record['trial_number']:
            # trial_record_clean.loc[itrial] = trial_record[trial_record['trial_number']==itrial+1].iloc[[0]]
            trial_record_clean = trial_record_clean.append(trial_record[trial_record['trial_number']==itrial].iloc[[0]])
        trial_record_clean = trial_record_clean.reset_index(drop = True)

        # change bhv_data time to the absolute time
        time_points_new = pd.DataFrame(np.zeros(np.shape(bhv_data)[0]),columns=["time_points_new"])
        # for itrial in np.arange(0,np.max(trial_record_clean['trial_number']),1):
        for itrial in np.arange(0,np.shape(trial_record_clean)[0],1):
            # ind = bhv_data["trial_number"]==itrial+1
            ind = bhv_data["trial_number"]==trial_record_clean['trial_number'][itrial]
            new_time_itrial = bhv_data[ind]["time_points"] + trial_record_clean["trial_starttime"].iloc[itrial]
            time_points_new["time_points_new"][ind] = new_time_itrial
        bhv_data["time_points"] = time_points_new["time_points_new"]
        bhv_data = bhv_data[bhv_data["time_points"] != 0]

        
        
        # load behavioral event results
        try:
            # dummy
            print('load social gaze with '+cameraID+' only of '+date_tgt)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_look_ornot.pkl', 'rb') as f:
                output_look_ornot = pickle.load(f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allvectors.pkl', 'rb') as f:
                output_allvectors = pickle.load(f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allangles.pkl', 'rb') as f:
                output_allangles = pickle.load(f)  
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_key_locations.pkl', 'rb') as f:
                output_key_locations = pickle.load(f)
        except:   

            # folder and file path
            if not do_OFC:
                camera12_analyzed_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/test_video_cooperative_task_DLPFCs_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera12/"
                camera23_analyzed_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/test_video_cooperative_task_DLPFCs_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera23/"
            elif do_OFC:
                camera12_analyzed_path = "/gpfs/marilyn/pi/nandy/VideoTracker_SocialInter/test_video_cooperative_task_OFCs_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera12/"
                camera23_analyzed_path = "/gpfs/marilyn/pi/nandy/VideoTracker_SocialInter/test_video_cooperative_task_OFCs_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera23/"

            # 
            try: 
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_167500"
                bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
                if not os.path.exists(bodyparts_camI_camIJ):
                    singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_80000"
                    bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
                if not os.path.exists(bodyparts_camI_camIJ):
                    singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_cameraSep1shuffle1_150000"
                    bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"                
                # get the bodypart data from files
                bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,videodate_tgt)
                video_file_original = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"
            except:
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_167500"
                bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
                if not os.path.exists(bodyparts_camI_camIJ):
                    singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_80000"
                    bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
                if not os.path.exists(bodyparts_camI_camIJ):
                    singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_cameraSep1shuffle1_150000"
                    bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            
            # get the bodypart data from files
            bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,videodate_tgt)
            video_file_original = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"        
        
            
            print('analyze social gaze with '+cameraID+' only of '+date_tgt)
            # get social gaze information 
            output_look_ornot, output_allvectors, output_allangles = find_socialgaze_timepoint_singlecam_wholebody(bodyparts_locs_camI,lever_locs_camI,tube_locs_camI,
                                                                                                                   considerlevertube,considertubeonly,sqr_thres_tubelever,
                                                                                                                   sqr_thres_face,sqr_thres_body)
            output_key_locations = find_socialgaze_timepoint_singlecam_wholebody_2(bodyparts_locs_camI,lever_locs_camI,tube_locs_camI,considerlevertube)
            
            # save data
            current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody/'+animal1_fixedorder[0]+animal2_fixedorder[0]
            add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
            if not os.path.exists(add_date_dir):
                os.makedirs(add_date_dir)
            #
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_look_ornot.pkl', 'wb') as f:
                pickle.dump(output_look_ornot, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allvectors.pkl', 'wb') as f:
                pickle.dump(output_allvectors, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allangles.pkl', 'wb') as f:
                pickle.dump(output_allangles, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_key_locations.pkl', 'wb') as f:
                pickle.dump(output_key_locations, f)
                

        look_at_other_or_not_merge = output_look_ornot['look_at_other_or_not_merge']
        look_at_tube_or_not_merge = output_look_ornot['look_at_tube_or_not_merge']
        look_at_lever_or_not_merge = output_look_ornot['look_at_lever_or_not_merge']
        look_at_otherlever_or_not_merge = output_look_ornot['look_at_otherlever_or_not_merge']
        look_at_otherface_or_not_merge = output_look_ornot['look_at_otherface_or_not_merge']
        
        # change the unit to second and align to the start of the session
        session_start_time = session_start_times[idate]
        look_at_other_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_other_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_lever_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_lever_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_tube_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_tube_or_not_merge['dodson'])[0],1)/fps - session_start_time 
        look_at_otherlever_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_otherlever_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_otherface_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_otherface_or_not_merge['dodson'])[0],1)/fps - session_start_time

        
        # find time point of behavioral events
        output_time_points_socialgaze ,output_time_points_levertube = bhv_events_timepoint_singlecam(bhv_data,look_at_other_or_not_merge,look_at_lever_or_not_merge,look_at_tube_or_not_merge)
        time_point_pull1 = output_time_points_socialgaze['time_point_pull1']
        time_point_pull2 = output_time_points_socialgaze['time_point_pull2']
        oneway_gaze1 = output_time_points_socialgaze['oneway_gaze1']
        oneway_gaze2 = output_time_points_socialgaze['oneway_gaze2']
        mutual_gaze1 = output_time_points_socialgaze['mutual_gaze1']
        mutual_gaze2 = output_time_points_socialgaze['mutual_gaze2']
        lever_gaze1 = output_time_points_levertube['time_point_lookatlever1']
        lever_gaze2 = output_time_points_levertube['time_point_lookatlever2']
        # 
        # mostly just for the sessions in which MC and SR are in the same session 
        firstpulltime = np.nanmin([np.nanmin(time_point_pull1),np.nanmin(time_point_pull2)])
        oneway_gaze1 = oneway_gaze1[oneway_gaze1>(firstpulltime-15)] # 15s before the first pull (animal1 or 2) count as the active period
        oneway_gaze2 = oneway_gaze2[oneway_gaze2>(firstpulltime-15)]
        mutual_gaze1 = mutual_gaze1[mutual_gaze1>(firstpulltime-15)]
        mutual_gaze2 = mutual_gaze2[mutual_gaze2>(firstpulltime-15)]  
        lever_gaze1 = lever_gaze1[lever_gaze1>(firstpulltime-15)]
        lever_gaze2 = lever_gaze2[lever_gaze2>(firstpulltime-15)]
        #    
        # newly added condition: only consider gaze during the active pulling time (15s after the last pull)    
        lastpulltime = np.nanmax([np.nanmax(time_point_pull1),np.nanmax(time_point_pull2)])
        oneway_gaze1 = oneway_gaze1[oneway_gaze1<(lastpulltime+15)]    
        oneway_gaze2 = oneway_gaze2[oneway_gaze2<(lastpulltime+15)]
        mutual_gaze1 = mutual_gaze1[mutual_gaze1<(lastpulltime+15)]
        mutual_gaze2 = mutual_gaze2[mutual_gaze2<(lastpulltime+15)] 
        lever_gaze1 = lever_gaze1[lever_gaze1<(lastpulltime+15)] 
        lever_gaze2 = lever_gaze2[lever_gaze2<(lastpulltime+15)] 
            
        # define successful pulls and failed pulls
        # a new definition of successful and failed pulls
        # separate successful and failed pulls
        # step 1 all pull and juice
        time_point_pull1 = bhv_data["time_points"][bhv_data["behavior_events"]==1]
        time_point_pull2 = bhv_data["time_points"][bhv_data["behavior_events"]==2]
        time_point_juice1 = bhv_data["time_points"][bhv_data["behavior_events"]==3]
        time_point_juice2 = bhv_data["time_points"][bhv_data["behavior_events"]==4]
        # step 2:
        # pull 1
        # Find the last pull before each juice
        successful_pull1 = [time_point_pull1[time_point_pull1 < juice].max() for juice in time_point_juice1]
        # Convert to Pandas Series
        successful_pull1 = pd.Series(successful_pull1, index=time_point_juice1.index)
        # Find failed pulls (pulls that are not successful)
        failed_pull1 = time_point_pull1[~time_point_pull1.isin(successful_pull1)]
        # pull 2
        # Find the last pull before each juice
        successful_pull2 = [time_point_pull2[time_point_pull2 < juice].max() for juice in time_point_juice2]
        # Convert to Pandas Series
        successful_pull2 = pd.Series(successful_pull2, index=time_point_juice2.index)
        # Find failed pulls (pulls that are not successful)
        failed_pull2 = time_point_pull2[~time_point_pull2.isin(successful_pull2)]
        #
        # step 3:
        time_point_pull1_succ = np.round(successful_pull1,1)
        time_point_pull2_succ = np.round(successful_pull2,1)
        time_point_pull1_fail = np.round(failed_pull1,1)
        time_point_pull2_fail = np.round(failed_pull2,1)
        # 
        time_point_pulls_succfail = { "pull1_succ":time_point_pull1_succ,
                                      "pull2_succ":time_point_pull2_succ,
                                      "pull1_fail":time_point_pull1_fail,
                                      "pull2_fail":time_point_pull2_fail,
                                    }
        
        # 
        # based on time point pull and juice, define some features for each pull action
        pull_infos = get_pull_infos(animal1, animal2, time_point_pull1, time_point_pull2, 
                                    time_point_juice1, time_point_juice2)
        
        # new total session time (instead of 600s) - total time of the video recording
        totalsess_time = np.ceil(np.shape(output_look_ornot['look_at_lever_or_not_merge']['dodson'])[0]/30) 
        #
        # remove task irrelavant period
        if totalsess_time > (lastpulltime+session_start_time+15):
            totalsess_time = np.ceil(lastpulltime+session_start_time+15)
        
        
        #
        # organize variables that are required by the HDDM functions
        # load the data first, if not process and then save the data 
        #
        # load the data that is organized for GLM, the goal is to do the GLM with the combined dataset across session
        try:
            dummy
            
            print('load the session wised data for GLM fitting')
            
            current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody_with_glm_model/'+animal1_fixedorder[0]+animal2_fixedorder[0]
            add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
            
            with open(add_date_dir+'/pre_data_for_GLM.pkl', 'rb') as f:
                pre_data_for_GLM = pickle.load(f)
        
        except:
            print('no sesison wise data saved for GLM, creating them now')
            #
            # MODIFICATION: Define kernel parameters here for easy adjustment
            KERNEL_DURATION_S = 4.0  # The length of the history kernel in seconds
            N_BASIS_FUNCS = 10       # The number of basis functions to represent the kernel
            
            # try:
            pre_data_for_GLM = continuous_variable_create_data_forGLM(KERNEL_DURATION_S, N_BASIS_FUNCS, fps, animal1, animal2, session_start_time,
                                                   time_point_pull1, time_point_pull2, oneway_gaze1, oneway_gaze2, 
                                                   mutual_gaze1, mutual_gaze2, animalnames_videotrack, 
                                                   output_look_ornot, output_allvectors, output_allangles, output_key_locations)
            # except:
            #     pre_data_for_GLM = np.nan
                
            #
            # save data
            if 1:
                current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody_with_glm_model/'+animal1_fixedorder[0]+animal2_fixedorder[0]
                add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
                if not os.path.exists(add_date_dir):
                    os.makedirs(add_date_dir)
                #
                with open(add_date_dir+'/pre_data_for_GLM.pkl', 'wb') as f:
                    pickle.dump(pre_data_for_GLM, f)
        #    
        pre_data_for_GLM_alldates[date_tgt] = pre_data_for_GLM
            
        
        # do the GLM session by session
        try:
            dummy
            print('load the result from GLM')
            
            current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody_with_glm_model/'+animal1_fixedorder[0]+animal2_fixedorder[0]
            add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
            
            with open(add_date_dir+'/glm_data.pkl', 'rb') as f:
                glm_data = pickle.load(f)
            with open(add_date_dir+'/glm_datas_shortlist_prediction.pkl', 'rb') as f:
                glm_datas_shortlist_prediction = pickle.load(f)
            
        except:
            print('no GLM data, analyze it and save it')
            #
            # MODIFICATION: Define kernel parameters here for easy adjustment
            KERNEL_DURATION_S = 4.0  # The length of the history kernel in seconds
            N_BASIS_FUNCS = 10       # The number of basis functions to represent the kernel
            
            try:
                glm_data = continuous_variable_glm(KERNEL_DURATION_S, N_BASIS_FUNCS, fps, animal1, animal2, session_start_time,
                                                   time_point_pull1, time_point_pull2, oneway_gaze1, oneway_gaze2, 
                                                   mutual_gaze1, mutual_gaze2, animalnames_videotrack, 
                                                   output_look_ornot, output_allvectors, output_allangles, output_key_locations)
            except:
                glm_data = np.nan
                
            try:
                glm_datas_shortlist_prediction = continuous_variable_glm_shortlist_prediction(KERNEL_DURATION_S, N_BASIS_FUNCS, 
                                                   fps, animal1, animal2, session_start_time,
                                                   time_point_pull1, time_point_pull2, oneway_gaze1, oneway_gaze2, 
                                                   mutual_gaze1, mutual_gaze2, animalnames_videotrack, 
                                                   output_look_ornot, output_allvectors, output_allangles, output_key_locations)
            except:     
                glm_datas_shortlist_prediction = np.nan
            
            #
            # save data
            if 1:
                current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody_with_glm_model/'+animal1_fixedorder[0]+animal2_fixedorder[0]
                add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
                if not os.path.exists(add_date_dir):
                    os.makedirs(add_date_dir)
                #
                with open(add_date_dir+'/glm_data.pkl', 'wb') as f:
                    pickle.dump(glm_data, f)
                    
                with open(add_date_dir+'/glm_datas_shortlist_prediction.pkl', 'wb') as f:
                    pickle.dump(glm_datas_shortlist_prediction, f)
      
        #    
        glm_datas_all_dates[date_tgt] = glm_data
        glm_datas_shortlist_prediction_all_dates[date_tgt] = glm_datas_shortlist_prediction
        

    # save data
    if 1:
        data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_glm'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
        if not os.path.exists(data_saved_subfolder):
            os.makedirs(data_saved_subfolder)

        with open(data_saved_subfolder+'/glm_datas_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(glm_datas_all_dates, f) 
            
        with open(data_saved_subfolder+'/glm_datas_shortlist_prediction_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(glm_datas_shortlist_prediction_all_dates, f) 
            
        with open(data_saved_subfolder+'/pre_data_for_GLM_alldates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(pre_data_for_GLM_alldates, f) 

    # only save a subset of data
    if 0:
        data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_glm'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
        if not os.path.exists(data_saved_subfolder):
            os.makedirs(data_saved_subfolder)
            
        with open(data_saved_subfolder+'/pre_data_for_GLM_alldates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(pre_data_for_GLM_alldates, f) 
    
    

In [ ]:
# analyze the target condition

if 1:
    #
    act_animal_to_ana = 'kanga'
    # act_animal_to_ana = 'dodson'
    
    act_animal_to_ana_partner = act_animal_to_ana+'_partner'
    act_animal_to_ana_backup = act_animal_to_ana
    
    #
    ###
    # For Kanga
    conditions_to_ana = ['MC', 'MC_withDodson','MC_withGinger', 'MC_withKoala', 'MC_withVermelho', ] # all MC
    # conditions_to_ana = ['SR', 'SR_withDodson', ] # all SR
    # conditions_to_ana = ['MC', 'MC_withDodson', 'MC_withVermelho', ] # MC with male
    # conditions_to_ana = ['MC_withGinger', 'MC_withKoala', ] # MC with female
    # conditions_to_ana = ['MC', ] # MC with familiar male
    # conditions_to_ana = ['MC_withGinger', ] # MC with familiar female
    # conditions_to_ana = ['MC_withDodson', 'MC_withVer|melho', ] # MC with unfamiliar male
    # conditions_to_ana = ['MC_withKoala', ] # MC with unfamiliar female
    # conditions_to_ana = ['MC_DannonAuto'] # partner AL
    # conditions_to_ana = ['MC_KangaAuto'] # self AL
    # conditions_to_ana = ['NV','NV_withDodson'] # NV
    # conditions_to_ana = ['MC', 'MC_withDodson','MC_withGinger', 'MC_withKoala', 'MC_withVermelho', 
    #                      'SR', 'SR_withDodson',]
    ###
    # For Dodson
    # conditions_to_ana = ['MC', 'MC_withGingerNew', 'MC_withKanga', 'MC_withKoala', ] # all MC
    # conditions_to_ana = ['MC', 'MC_withKanga', ] # all MC
    # conditions_to_ana = ['SR', 'SR_withGingerNew', 'SR_withKanga', 'SR_withKoala', ] # all SR
    # conditions_to_ana = ['MC', 'MC_withKanga', 'MC_withKoala', ] # all MC, no gingerNew
    # conditions_to_ana = ['SR', 'SR_withKanga', 'SR_withKoala', ] # all SR,  no gingerNew
    # conditions_to_ana = ['MC', 'MC_withGingerNew', 'MC_withKanga', 'MC_withKoala', ] # MC with female
    # conditions_to_ana = ['MC', 'MC_withGingerNew', ] # MC with familiar female
    # conditions_to_ana = ['MC_withKanga', 'MC_withKoala', ] # MC with unfamiliar female
    # conditions_to_ana = ['MC_KoalaAuto_withKoala'] # partner AL
    # conditions_to_ana = ['MC_DodsonAuto_withKoala'] # self AL
    # conditions_to_ana = ['NV_withKanga'] # NV
    # conditions_to_ana = ['MC', 'MC_withGingerNew', 'MC_withKanga', 'MC_withKoala', 
    #                      'SR', 'SR_withGingerNew', 'SR_withKanga', 'SR_withKoala', ]

    cond_toplot_type = 'allMC'

#####
# for answer R2's question

# looking at the partner's of kanga or dodson

if 0:
    # for Kanga
    
    # dannon
    act_animal_to_ana = 'dannon'
    conditions_to_ana = ['MC']
    
    # # dodson
    # act_animal_to_ana = 'dodson'
    # conditions_to_ana = ['MC_withDodson'] 
    
    # # ginger
    # act_animal_to_ana = 'ginger'
    # conditions_to_ana = ['MC_withGinger']
    
    # # koala
    # act_animal_to_ana = 'koala'
    # conditions_to_ana = ['MC_withKoala']
    
    # # vermelho
    # act_animal_to_ana = 'vermelho'
    # conditions_to_ana = ['MC_withVermelho']
    
    #############
    # for Dodson
    
    # # for ginger
    # act_animal_to_ana = 'ginger' 
    # conditions_to_ana = ['MC', 'MC_withGingerNew']
    # # for kanga
    # act_animal_to_ana = 'kanga' 
    # conditions_to_ana = ['MC_withKanga']
    # # for koala
    # act_animal_to_ana = 'koala' 
    # conditions_to_ana = ['MC_withKoala']

    
    cond_toplot_type = 'allMC_partnerfocus'



In [ ]:
# get the target data and plot the summarizing figure for fitting performance
if 0:
    ind_tgt = np.isin(task_conditions,conditions_to_ana)

    dates_tgt = np.array(dates_list)[ind_tgt]
    ndates = np.shape(dates_tgt)[0]

    # 
    mean_beta_df_all = []

    for idate in np.arange(0,ndates,1):

        date_tgt = dates_tgt[idate]

        try:
            mean_beta_df_tgt = glm_datas_all_dates[date_tgt][(act_animal_to_ana,'mean_beta_df')]
            mean_beta_df_tgt['date'] = date_tgt

            mean_beta_df_all.append(mean_beta_df_tgt)
        except:
            continue
    #
    mean_beta_df_all = pd.concat(mean_beta_df_all, ignore_index=True)


    #
    # set up and do the plotting
    #
    # Set the desired variable order
    var_names = [
        'gaze_other_angle', 'gaze_tube_angle', 'gaze_lever_angle',
        'animal_animal_dist', 'animal_tube_dist', 'animal_lever_dist',
        'mass_move_speed', 'gaze_angle_speed'
    ]    

    # Ensure the necessary column is available
    mean_beta_df_all['neg_log10_p'] = -np.log10(mean_beta_df_all['LRT_pvalue'])
    mean_beta_df_all['is_significant'] = mean_beta_df_all['LRT_pvalue'] < 0.05
    mean_beta_df_all['Variable'] = pd.Categorical(mean_beta_df_all['Variable'], categories=var_names, ordered=True)

    # === FIGURE 1: Violin plot of -log10(p-value) ===
    plt.figure(figsize=(12, 6))
    sns.violinplot(
        data=mean_beta_df_all,
        x='Variable',
        y='neg_log10_p',
        order=var_names,
        inner='point',
        scale='width',
        palette='Set2'
    )
    plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='p = 0.05')
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('-log10(LRT p-value)')
    plt.title('Figure 1: Distribution of Significance (-log10 p) Across Sessions')
    plt.legend()
    plt.tight_layout()

    # === FIGURE 2: Bar plot of % sessions with significant LRT ===
    sig_summary = mean_beta_df_all.groupby('Variable')['is_significant'].mean().reset_index()
    sig_summary['percentage'] = sig_summary['is_significant'] * 100

    plt.figure(figsize=(12, 6))
    sns.barplot(data=sig_summary, x='Variable', y='percentage', palette='Set2',order=var_names,)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('% Sessions with Significant LRT (p < 0.05)')
    plt.title('Figure 2: Frequency of Significant LRT Results per Variable')
    plt.tight_layout()



In [ ]:
# get the target data and plot the summarizing figure for fitting performance
# focus on the prediction accuracy / roauc
if 0:
    ind_tgt = np.isin(task_conditions,conditions_to_ana)

    dates_tgt = np.array(dates_list)[ind_tgt]
    ndates = np.shape(dates_tgt)[0]
    
    # 
    glm_prediction_auc_all = pd.DataFrame(columns=['date','mean_auc_full','mean_auc_short','mean_auc_other'])

    
    for idate in np.arange(0,ndates,1):

        date_tgt = dates_tgt[idate]

        try:
            mean_auc_full = np.nanmean(glm_datas_shortlist_prediction_all_dates[date_tgt]\
                                            [(act_animal_to_ana,'predictive_perf')]['auc_full'])
            mean_auc_short = np.nanmean(glm_datas_shortlist_prediction_all_dates[date_tgt]\
                                            [(act_animal_to_ana,'predictive_perf')]['auc_short'])
            mean_auc_other = np.nanmean(glm_datas_shortlist_prediction_all_dates[date_tgt]\
                                            [(act_animal_to_ana,'predictive_perf')]['auc_other'])
            
            row_data = {
                    'date': date_tgt,
                    'mean_auc_full': mean_auc_full,
                    'mean_auc_short': mean_auc_short,
                    'mean_auc_other': mean_auc_other,
                        }
            glm_prediction_auc_all = glm_prediction_auc_all.append(
                                            row_data, ignore_index=True)
        except:
            continue
    
    
    from scipy.stats import ttest_rel

    # Paired t-test
    tstat, pval = ttest_rel(glm_prediction_auc_all['mean_auc_full'], glm_prediction_auc_all['mean_auc_short'])

    # Compute mean difference
    mean_other = glm_prediction_auc_all['mean_auc_full'].mean()
    mean_short = glm_prediction_auc_all['mean_auc_short'].mean()
    mean_diff = mean_other - mean_short

    # Melt data for violin plot
    df_melt = glm_prediction_auc_all[['mean_auc_full', 'mean_auc_short']].melt(var_name='Model', value_name='AUC')

    # Plot
    plt.figure(figsize=(6, 6))
    sns.violinplot(data=df_melt, x='Model', y='AUC', inner='box', palette='pastel')
    sns.swarmplot(data=df_melt, x='Model', y='AUC', color='k', size=4)

    # Annotate p-value and mean diff
    x1, x2 = 0, 1
    y, h, col = df_melt['AUC'].max() + 0.02, 0.01, 'k'
    plt.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.5, c=col)
    plt.text((x1 + x2) * .5, y + h + 0.01, f"*p = {pval:.3f}", ha='center', va='bottom', color=col)
    plt.text((x1 + x2) * .5, y + h + 0.04, f"Δ = {mean_diff:.3f}", ha='center', va='bottom', color='blue')

    # Labels and formatting
    plt.title('AUC Comparison: Other vs Short Model')
    plt.xlim([-0.5, 1.5])
    plt.ylim([0.6, 1.1])
    plt.xticks([0, 1], ['Other', 'Short'])

    # Save figure
    savefig = 0
    if savefig:
        if not do_OFC:
            figsavefolder = data_saved_folder + "fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_glm_model/" + \
                        cameraID + "/" + animal1_filenames[0] + "_" + animal2_filenames[0] + "/glm_fitting_summary_fig/"
        elif do_OFC:
            figsavefolder = data_saved_folder + "fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_glm_model_OFC/" + \
                        cameraID + "/" + animal1_filenames[0] + "_" + animal2_filenames[0] + "/glm_fitting_summary_fig/"
            
        if not os.path.exists(figsavefolder):
            os.makedirs(figsavefolder)

        plt.savefig(figsavefolder + act_animal_to_ana + '_in_' + cond_toplot_type +
                    '_glm_fitting_summary_other_vs_short_figure.pdf')
    
    

In [ ]:
# combine the raw data together across session (in the same or tgt condition) and then run the glm
# the goal is to test which variable contribute the most

# this is the main analysis for the paper, turn off for now 
if 0:
    
    N_BASIS_FUNCS = 10
    
    import seaborn as sns
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score
    from group_lasso import LogisticGroupLasso
    
    
    ind_tgt = np.isin(task_conditions,conditions_to_ana)

    dates_tgt = np.array(dates_list)[ind_tgt]
    ndates = np.shape(dates_tgt)[0]

    # 
    X_all_conbined = []
    Y_all_conbined = []
    
    for idate in np.arange(0,ndates,1):

        date_tgt = dates_tgt[idate]
        
        var_names = pre_data_for_GLM_alldates[date_tgt][(act_animal_to_ana,'var_names')]
        
        X_idate = pre_data_for_GLM_alldates[date_tgt][(act_animal_to_ana,'X_all')]
        Y_idate = pre_data_for_GLM_alldates[date_tgt][(act_animal_to_ana,'Y')]
        
        # Z-score normalize per session
        scaler = StandardScaler()
        X_idate_zscored = scaler.fit_transform(X_idate)

        # Optional: remove nan rows
        valid_rows = ~np.isnan(X_idate_zscored).any(axis=1)
        X_valid = X_idate_zscored[valid_rows]
        Y_valid = Y_idate[valid_rows]

        X_all_conbined.append(X_valid)
        Y_all_conbined.append(Y_valid)
        
    
    X_combined = np.vstack(X_all_conbined)
    Y_combined = np.concatenate(Y_all_conbined)
    
    # ==============================================================================
    # BOOTSTRAP WITH LEAVE-ONE-GROUP-OUT & GROUP LASSO
    # ==============================================================================

    # 1. Setup bootstrap and results storage
    n_bootstraps = 100
    results = {'full_model': []}
    for var in var_names:
        results[f'drop_{var}'] = []

    ## NEW ##
    # Add a list to store the Group Lasso coefficients from each iteration
    lasso_coeffs_all_runs = [] 
    # Define the feature groups once, as it's the same for all iterations
    n_variables = len(var_names)
    groups = np.repeat(np.arange(n_variables), repeats=N_BASIS_FUNCS)


    # 2. Start bootstrap loop
    for i in range(n_bootstraps):
        print(f"🚀 Running bootstrap iteration {i+1}/{n_bootstraps}...")

        # a. Balance dataset for this iteration
        # (Your existing balancing code is here)
        pos_idx = np.where(Y_combined == 1)[0]
        neg_idx = np.where(Y_combined == 0)[0]
        if len(pos_idx) == 0 or len(neg_idx) < len(pos_idx):
            print(f"Warning: Not enough samples for iteration {i+1}. Skipping.")
            continue
        neg_sample_idx = np.random.choice(neg_idx, size=len(pos_idx), replace=False)
        balanced_idx = np.concatenate([pos_idx, neg_sample_idx])
        np.random.shuffle(balanced_idx)
        X_balanced = X_combined[balanced_idx]
        Y_balanced = Y_combined[balanced_idx]

        # b. Split into train and test sets
        X_train, X_test, y_train, y_test = train_test_split(
            X_balanced, Y_balanced, test_size=0.2, stratify=Y_balanced
        )

        # c. Normalize features based ONLY on the training set
        scaler = StandardScaler()
        X_train_norm = scaler.fit_transform(X_train)
        X_test_norm = scaler.transform(X_test)

        # --- Leave-One-Out Analysis (Your existing code) ---
        # NOTE: I've removed `class_weight='balanced'` because you are already manually balancing the data. Using both is redundant.

        # d. Fit and evaluate the FULL model
        clf_full = LogisticRegression(max_iter=1000) 
        clf_full.fit(X_train_norm, y_train)
        y_proba_full = clf_full.predict_proba(X_test_norm)[:, 1]
        auc_full = roc_auc_score(y_test, y_proba_full)
        results['full_model'].append(auc_full)

        # e. Loop through each variable group to perform leave-one-group-out
        for j, var_to_drop in enumerate(var_names):
            start_col = j * N_BASIS_FUNCS
            end_col = start_col + N_BASIS_FUNCS
            cols_to_drop = np.arange(start_col, end_col)
            X_train_loo = np.delete(X_train_norm, cols_to_drop, axis=1)
            X_test_loo = np.delete(X_test_norm, cols_to_drop, axis=1)

            clf_loo = LogisticRegression(max_iter=1000)
            clf_loo.fit(X_train_loo, y_train)
            y_proba_loo = clf_loo.predict_proba(X_test_loo)[:, 1]
            auc_loo = roc_auc_score(y_test, y_proba_loo)
            results[f'drop_{var_to_drop}'].append(auc_loo)

        ## NEW ## 
        # --- Group Lasso Analysis (run on the same data split) ---
        gl_model = LogisticGroupLasso(
            groups=groups,
            group_reg=0.05,  # This is a key parameter to tune
            supress_warning=True
        )
        gl_model.fit(X_train_norm, y_train)
        # Store the resulting coefficients for this iteration
        lasso_coeffs_all_runs.append(gl_model.coef_)


    print("\n✅ Bootstrap analysis complete.")

    # 3. Summarize and Plot the results
    # --- First, analyze and plot the Leave-One-Out results (your existing code) ---
    results_df = pd.DataFrame(results)
    summary_stats = results_df.agg(['mean', 'std']).T
    
    summary_stats.rename(columns={'mean': 'mean_auc', 'std': 'std_auc'}, inplace=True)
    mean_full_auc = summary_stats.loc['full_model', 'mean_auc']
    summary_stats['auc_drop_from_full'] = mean_full_auc - summary_stats['mean_auc']

    print("\n--- 📊 Summary of Model Performance ---")
    print(summary_stats.sort_values(by='auc_drop_from_full', ascending=False))

    # Plotting...
    # (The plotting code from the previous answer can be used here without any changes)
    # --- Plot 1: Mean AUC for each model ---
    print("📊 Generating violin plot of model performance...")
    fig, ax = plt.subplots(figsize=(12, 7))
    # Reshape the DataFrame from wide to long format for Seaborn
    plot_df_long = results_df.melt(var_name='model', value_name='auc')
    # Define the desired order to prevent automatic sorting
    model_order = ['full_model'] + [f'drop_{var}' for var in var_names]
    # Create the violin plot
    sns.violinplot(
        data=plot_df_long, 
        x='model', 
        y='auc', 
        order=model_order, 
        inner='quartile', # Shows the quartiles inside the violins
        palette='viridis',
        ax=ax
    )
    # Overlay individual data points for more detail
    sns.stripplot(
        data=plot_df_long,
        x='model',
        y='auc',
        order=model_order,
        size=2,
        color="white",
        edgecolor='gray',
        ax=ax
    )
    ax.set_ylabel('ROC AUC Score Distribution', fontsize=14)
    ax.set_xlabel('Model Type', fontsize=14)
    ax.set_title(f'Model Performance Distribution ({n_bootstraps} Bootstraps)', fontsize=16)
    ax.axhline(y=0.5, color='black', linestyle='--', label='Chance Level (AUC = 0.5)')
    ax.legend()
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()

    # --- Plot 2: Feature importance with error bars ---
    xvarnames_tosort = ['gaze_other_angle','gaze_tube_angle','gaze_lever_angle','animal_animal_dist',
                        'animal_tube_dist','animal_lever_dist','mass_move_speed','gaze_angle_speed']
    #
    # 1. Calculate the AUC drop for each bootstrap run to get a distribution of importance scores.
    auc_drops_df = pd.DataFrame()
    for col in results_df.columns:
        if 'drop_' in col:
            auc_drops_df[col] = results_df['full_model'] - results_df[col]
    # 2. Calculate the mean and standard deviation from these distributions.
    mean_drops = auc_drops_df.mean()
    std_drops = auc_drops_df.std()
    sem_drops = std_drops/np.sqrt(n_bootstraps)
    # 3. Create a new DataFrame for plotting.
    importance_data = pd.DataFrame({
        'mean_drop': mean_drops,
        'std_drop': std_drops,
        'sem_drop': sem_drops,
    })
    #
    # Create a correctly prefixed list for sorting
    sorted_index = ['drop_' + name for name in xvarnames_tosort]
    # Reorder the DataFrame based on your list
    importance_data = importance_data.reindex(sorted_index)
    # 4. Create the plot.
    fig2, ax2 = plt.subplots(figsize=(12, 8))
    # Switched to a vertical bar plot
    ax2.bar(
        importance_data.index.str.replace('drop_', ''),
        importance_data['mean_drop'],
        # Switched to yerr for vertical error bars
        yerr=importance_data['sem_drop'],
        capsize=4, # Adds caps to the error bars
        color='lightgreen',
        edgecolor='black'
    )
    # Swapped axis labels
    ax2.set_ylabel('Drop in Mean AUC (Importance)', fontsize=14)
    ax2.set_xlabel('Variable Group Removed', fontsize=14)
    ax2.set_title('Variable Importance based on Performance Drop', fontsize=16)
    # Switched to a horizontal line
    ax2.axhline(y=0, color='grey', linestyle='--')
    # Added rotation for better label readability
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()

    ## NEW ##
    # --- Second, analyze and plot the Group Lasso results ---
    print("\n--- 📊 Summary of Group Lasso Coefficients ---")
    # Average the coefficients across all bootstrap runs
    mean_lasso_coeffs = np.mean(lasso_coeffs_all_runs, axis=0)

    # Calculate the average magnitude of coefficients for each variable group
    group_importance = []
    for i, var_name in enumerate(var_names):
        start_idx = i * N_BASIS_FUNCS
        end_idx = start_idx + N_BASIS_FUNCS
        # Use the absolute mean coefficient magnitude as the importance score
        importance_score = np.mean(np.abs(mean_lasso_coeffs[start_idx:end_idx]))
        group_importance.append({'variable': var_name, 'importance': importance_score})

    # Create a DataFrame for easy sorting and plotting
    lasso_importance_df = pd.DataFrame(group_importance).sort_values('importance', ascending=True)

    print(lasso_importance_df)

    # Plot 3: Group Lasso Feature Importance
    fig3, ax3 = plt.subplots(figsize=(10, 8))
    ax3.barh(
        lasso_importance_df['variable'],
        lasso_importance_df['importance'],
        color='purple',
        edgecolor='black'
    )
    ax3.set_xlabel('Mean Absolute Coefficient (Importance)', fontsize=14)
    ax3.set_ylabel('Variable Group', fontsize=14)
    ax3.set_title('Variable Importance from Group Lasso', fontsize=16)
    plt.tight_layout()
    
    # Save figure
    savefig = 1
    if savefig:
        if not do_OFC:
            figsavefolder = data_saved_folder + "fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_glm_model/" + \
                        cameraID + "/" + animal1_filenames[0] + "_" + animal2_filenames[0] + "/glm_fitting_summary_fig/"
        elif do_OFC:
            figsavefolder = data_saved_folder + "fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_glm_model_OFC/" + \
                        cameraID + "/" + animal1_filenames[0] + "_" + animal2_filenames[0] + "/glm_fitting_summary_fig/"
            
        if not os.path.exists(figsavefolder):
            os.makedirs(figsavefolder)

        fig.savefig(figsavefolder + act_animal_to_ana + '_in_' + cond_toplot_type +
                    '_glm_fitting_auc_full_model_and_leave_one_variable_out.pdf')
        
        fig2.savefig(figsavefolder + act_animal_to_ana + '_in_' + cond_toplot_type +
                    '_glm_fitting_drop_in_auc_full_model_and_leave_one_variable_out.pdf')
        
        fig3.savefig(figsavefolder + act_animal_to_ana + '_in_' + cond_toplot_type +
                    '_glm_fitting_group_lasso_full_model_.pdf')

        

In [ ]:
# reconstruct the score/weight that predict pull

if 0:
    
    import statsmodels.api as sm
    
    timepoints = np.arange(-4,4,1/fps)
    
    ind_tgt = np.isin(task_conditions,conditions_to_ana)

    dates_tgt = np.array(dates_list)[ind_tgt]
    ndates = np.shape(dates_tgt)[0]

    # Initialize storage containers outside the loop
    all_session_scores = {}
    all_pull_snapshots = {}

    # --- Inside your existing loop ---
    for idate in np.arange(0, ndates, 1):
        date_tgt = dates_tgt[idate]
    
        try:
            # Retrieve objects from your existing dictionary structure
            all_betas_df = glm_datas_all_dates[date_tgt][(act_animal_to_ana, 'summary_df')]
            all_variables = pre_data_for_GLM_alldates[date_tgt][(act_animal_to_ana, 'X_all')]
            all_pulls = pre_data_for_GLM_alldates[date_tgt][(act_animal_to_ana, 'Y')]
        except:
            continue

        # 1. Prepare Design Matrix for scoring
        # Ensure you use the exact same scaler used during training
        X_scaled = scaler.transform(all_variables) 
        X_design = sm.add_constant(X_scaled, has_constant='add')

        # 2. Calculate Continuous Score (Full session)
        # weights: [intercept, b1_var1, b2_var1 ... bN_varM]
        weights = all_betas_df['Coefficient'].values
        full_session_score = np.dot(X_design, weights)
        all_session_scores[date_tgt] = full_session_score

        # 3. Extract Pull-Aligned Snapshots (-4s to 4s)
        pull_indices = np.where(all_pulls == 1)[0]
        window_frames = int(4 * fps)

        date_pull_matrix = []
        for idx in pull_indices:
            start = idx - window_frames
            end = idx + window_frames

            # Ensure window is within session bounds
            if start >= 0 and end <= len(full_session_score):
                date_pull_matrix.append(full_session_score[start:end])

        # Convert to numpy array: [n_pulls, 8 * fps]
        all_pull_snapshots[date_tgt] = np.array(date_pull_matrix)

        print(f"Processed {date_tgt}: {len(date_pull_matrix)} pulls identified.")
        
        date_pull_matrix = all_pull_snapshots[date_tgt]

        if date_pull_matrix.shape[0] > 1: # Only plot if we have more than one pull
            # Calculate Mean and Std across the pulls (axis 0)
            mean_score = np.mean(date_pull_matrix, axis=0)
            std_score = np.std(date_pull_matrix, axis=0)

            # Define time axis
            time_axis = np.linspace(-4, 4, len(mean_score))

            # Plotting
            plt.figure(figsize=(8, 4))
            plt.plot(time_axis, mean_score, label='Mean Score', color='blue', lw=2)
            plt.fill_between(time_axis, 
                             mean_score - std_score, 
                             mean_score + std_score, 
                             color='blue', alpha=0.2, label='±1 SD')

            # Formatting
            plt.axvline(0, color='red', linestyle='--', label='Pull Onset')
            plt.title(f'Pull-Aligned Evidence Score: {date_tgt}')
            plt.xlabel('Time from pull (s)')
            plt.ylabel('Log-Odds Score')
            plt.legend()
            plt.grid(True, alpha=0.3)

            # Save or show
            plt.savefig(f"pull_score_plot_{date_tgt}.png")
            plt.show()
        else:
            print(f"Insufficient pulls for {date_tgt} to calculate STD.")
        
        
           

In [ ]:
# reconstruct the score/weight that predict pull
# compare with the neuron firing rate PCs
# 

if 0:
    
    import statsmodels.api as sm
    from scipy.stats import ttest_1samp
    from scipy.stats import pearsonr
    import seaborn as sns
    
    
    def get_sem(data, axis=0):
        return np.nanstd(data, axis=axis) / np.sqrt(data.shape[axis])
    
    # --- Helper to remove NaNs across both datasets ---
    def filter_nans(score_mat, pc_mat):
        # Check for NaNs in either the tracking matrix or the PC matrix
        # We look for NaNs along axis 1 (the time window)
        mask = ~np.isnan(score_mat).any(axis=1) & ~np.isnan(pc_mat).any(axis=1)
        return score_mat[mask], pc_mat[mask]
    
    #
    # load all firing rate PCs aligned at bhv
    data_saved_subfolder_2 = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
    with open(data_saved_subfolder_2+'/bhvevents_aligned_FRPCs_allevents_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        bhvevents_aligned_FRPCs_allevents_all_dates = pickle.load(f)
    
    timepoints = np.arange(-4,4,1/fps)
    
    ind_tgt = np.isin(task_conditions,conditions_to_ana)

    dates_tgt = np.array(dates_list)[ind_tgt]
    ndates = np.shape(dates_tgt)[0]

    # Initialize storage containers outside the loop
    all_session_scores = {}
    all_pull_snapshots = {}
    all_ensemble_corrs = {}
    all_r1, all_r2, all_r3 = [], [], []
    all_p1, all_p2, all_p3 = [], [], []
    all_best_r2 = []
    all_best_pc_labels = []
    

    # --- Inside your existing loop ---
    for idate in np.arange(0, ndates, 1):
        date_tgt = dates_tgt[idate]
        
        # Retrieve firing rate PCs
        # Ensure these are shapes (n_pulls, time_window)
        FRPC1 = bhvevents_aligned_FRPCs_allevents_all_dates[date_tgt][act_animal_to_ana+' pull']['pc1']['FR_allevents']
        FRPC2 = bhvevents_aligned_FRPCs_allevents_all_dates[date_tgt][act_animal_to_ana+' pull']['pc2']['FR_allevents']
        FRPC3 = bhvevents_aligned_FRPCs_allevents_all_dates[date_tgt][act_animal_to_ana+' pull']['pc3']['FR_allevents']

        try:
            all_betas_df = glm_datas_all_dates[date_tgt][(act_animal_to_ana, 'summary_df')]
            all_variables = pre_data_for_GLM_alldates[date_tgt][(act_animal_to_ana, 'X_all')]
            all_pulls = pre_data_for_GLM_alldates[date_tgt][(act_animal_to_ana, 'Y')]
        except:
            continue

        # 1. Prepare Design Matrix and Calculate Score
        X_scaled = scaler.transform(all_variables) 
        X_design = sm.add_constant(X_scaled, has_constant='add')
        weights = all_betas_df['Coefficient'].values

        full_session_score = np.dot(X_design, weights)
        all_session_scores[date_tgt] = full_session_score

        # 2. Extract Pull-Aligned Snapshots (-4s to 4s)
        pull_indices = np.where(all_pulls == 1)[0]
        window_frames = int(4 * fps)

        date_pull_matrix = []
        for idx in pull_indices:
            start = idx - window_frames
            end = idx + window_frames
            if start >= 0 and end <= len(full_session_score):
                date_pull_matrix.append(full_session_score[start:end])

        score_matrix = np.array(date_pull_matrix)
        all_pull_snapshots[date_tgt] = score_matrix
        
        # 1. Align the pull numbers first (take the intersection of available pulls)
        # We take the minimum count to ensure shapes match for broadcasting
        min_pulls = min(score_matrix.shape[0], FRPC1.T.shape[0])

        # Truncate all matrices to the minimum pull count
        s_mat = score_matrix[:min_pulls, :]
        f1, f2, f3 = FRPC1.T[:min_pulls, :], FRPC2.T[:min_pulls, :], FRPC3.T[:min_pulls, :]
        
        # Filter: remove any pull that has a NaN in any of the 4 matrices
        # (score_matrix, f1, f2, or f3)
        valid_mask = (~np.isnan(score_matrix).any(axis=1) & 
                      ~np.isnan(f1).any(axis=1) & 
                      ~np.isnan(f2).any(axis=1) & 
                      ~np.isnan(f3).any(axis=1))

        s_clean = score_matrix[valid_mask]
        f1_clean, f2_clean, f3_clean = f1[valid_mask], f2[valid_mask], f3[valid_mask]

        # 2. Calculate Correlations for all 3 PCs
        if 0: # the entire trace
            # Instead of just np.corrcoef, use pearsonr:
            # pearsonr returns (r, p-value)
            r1, p1 = pearsonr(s_clean.flatten(), f1_clean.flatten())
            r2, p2 = pearsonr(s_clean.flatten(), f2_clean.flatten())
            r3, p3 = pearsonr(s_clean.flatten(), f3_clean.flatten())
        #
        if 1: # the mean trace
            # Instead of flattening the whole matrix, take the mean across pulls (axis=0)
            # This compares the shape of the average GLM score to the average PC
            s_mean = np.mean(s_clean, axis=0)
            f1_mean = np.mean(f1_clean, axis=0)
            f2_mean = np.mean(f2_clean, axis=0)
            f3_mean = np.mean(f3_clean, axis=0)
            #
            r1, p1 = pearsonr(s_mean, f1_mean)
            r2, p2 = pearsonr(s_mean, f2_mean)
            r3, p3 = pearsonr(s_mean, f3_mean)
            
        # Now you have a p-value for every session!
        # You can store these to color your swarm plot:
        all_p1.append(p1)
        all_p2.append(p2)
        all_p3.append(p3)
        #
        # Store the R^2
        all_r1.append(r1**2)
        all_r2.append(r2**2)
        all_r3.append(r3**2)
        #
        r_sq_vals = [r1**2, r2**2, r3**2]
        
        # Find the maximum R^2 for this session
        best_r2 = np.max(r_sq_vals)
        all_best_r2.append(best_r2)
        
        # Identify which PC was the best (returns 0, 1, or 2, so we add 1)
        best_idx = np.argmax(r_sq_vals) + 1
        all_best_pc_labels.append(f"PC{best_idx}")
        

        # 3. Visualization
        if s_clean.shape[0] > 1:
            data_to_plot = {
                'GLM Score': s_clean,
                'PC1': f1_clean,
                'PC2': f2_clean,
                'PC3': f3_clean
            }

            fig, axes = plt.subplots(1, 4, figsize=(13, 4), sharex=False)
            corr_text = f"r1={r1:.2f}, r2={r2:.2f}, r3={r3:.2f}"

            for i, (name, matrix) in enumerate(data_to_plot.items()):
                mean_val = np.mean(matrix, axis=0)
                sem_val = get_sem(matrix, axis=0)
                time_axis = np.linspace(-4, 4, matrix.shape[1])

                axes[i].plot(time_axis, mean_val, lw=2, color='k')
                axes[i].fill_between(time_axis, mean_val - sem_val, mean_val + sem_val, alpha=0.3, color='k')
                axes[i].axvline(0, color='red', linestyle='--', alpha=0.7)
                axes[i].set_ylabel(name)
                axes[i].grid(True, alpha=0.3)

            axes[-1].set_xlabel('Time from pull (s)')
            plt.suptitle(f'Pull-Aligned Dynamics: {date_tgt}\n{corr_text}')
            plt.tight_layout(rect=[0, 0, 1, 0.95])
            plt.show()
    
    
    # --- Post-Loop Visualization ---
    # Create DataFrame for plotting
    corrs_df = pd.DataFrame({'PC1': all_r1, 'PC2': all_r2, 'PC3': all_r3})
    p_values_df = pd.DataFrame({'PC1': all_p1, 'PC2': all_p2, 'PC3': all_p3})

    # Melt data for seaborn
    plot_data = corrs_df.melt(var_name='PC', value_name='R2')
    p_values_flat = p_values_df.melt(var_name='PC', value_name='p')['p']

    # Define threshold and create a specific Hue column
    p_threshold = 0.01
    plot_data['Significance'] = ['Significant' if p < p_threshold else 'Not Significant' for p in p_values_flat]

    # Plotting
    plt.figure(figsize=(6, 5))

    # Swarm plot
    # We map 'hue' directly to our 'Significance' column
    sns.swarmplot(x='PC', y='R2', data=plot_data, size=8, 
                  hue='Significance', 
                  palette={'Significant': 'red', 'Not Significant': 'gray'})
    

    plt.ylabel(r'Variance Explained ($R^2$)')
    plt.title(f'Predictive Power of Neural PCs across Sessions\n(Red: p < {p_threshold} per session)')
    plt.grid(axis='y', alpha=0.3)

    # Move legend outside the plot so it doesn't cover data
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
        
        
    # Create DataFrame for plotting
    plot_data = pd.DataFrame({
        'Max_R2': all_best_r2,
        'Best_PC': all_best_pc_labels
    })

    plt.figure(figsize=(5, 6))

    # Swarm plot: Plotting the best R^2 per session
    # Hue colors the dot based on whether PC1, PC2, or PC3 was the winner that day
    sns.swarmplot(y='Max_R2', data=plot_data, 
                  hue='Best_PC', size=9, 
                  palette={'PC1': '#d62728', 'PC2': '#1f77b4', 'PC3': '#2ca02c'}, 
                  hue_order=['PC1', 'PC2', 'PC3'])

    # Add boxplot overlay for statistical summary
    sns.boxplot(y='Max_R2', data=plot_data, 
                showcaps=False, boxprops={'facecolor':'none', 'edgecolor':'black', 'alpha':0.5},
                showfliers=False, whiskerprops={'linewidth':0},
                medianprops={'color':'black', 'linewidth':2.5})

    plt.ylabel(r'Maximum Variance Explained ($R^2$)')
    plt.title('Predictive Power of the\nBest Neural Component per Session')
    plt.grid(axis='y', alpha=0.3)

    # Expand y-axis slightly to give the dots room to breathe
    plt.ylim(-0.05, plot_data['Max_R2'].max() + 0.1)

    # Format the legend
    plt.legend(title="Which PC won?", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

    # Print a quick summary to the console
    mean_best = np.mean(all_best_r2)
    print(f"Across all sessions, the best PC explains an average of {mean_best*100:.1f}% of the variance.")


In [ ]:
# reconstruct the score/weight that predict pull
# compare with the neuron firing rate for each neuron in each session
# 

if 1:
    
    import statsmodels.api as sm
    from scipy.stats import ttest_1samp
    from scipy.stats import pearsonr
    import seaborn as sns
    
    
    def get_sem(data, axis=0):
        return np.nanstd(data, axis=axis) / np.sqrt(data.shape[axis])
    
    # --- Helper to remove NaNs across both datasets ---
    def filter_nans(score_mat, pc_mat):
        # Check for NaNs in either the tracking matrix or the PC matrix
        # We look for NaNs along axis 1 (the time window)
        mask = ~np.isnan(score_mat).any(axis=1) & ~np.isnan(pc_mat).any(axis=1)
        return score_mat[mask], pc_mat[mask]
    
    #
    # load all firing rate PCs aligned at bhv
    data_saved_subfolder_2 = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
    with open(data_saved_subfolder_2+'/bhvevents_aligned_FR_allevents_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        bhvevents_aligned_FRs_allevents_all_dates = pickle.load(f)
    
    timepoints = np.arange(-4,4,1/fps)
    
    ind_tgt = np.isin(task_conditions,conditions_to_ana)

    dates_tgt = np.array(dates_list)[ind_tgt]
    ndates = np.shape(dates_tgt)[0]

    # Initialize storage containers outside the loop
    all_session_scores = {}
    all_pull_snapshots = {}
    all_ensemble_corrs = {}
    all_r1 = []
    all_p1 = []
    

    # --- Inside your existing loop ---
    for idate in np.arange(0, ndates, 1):
        date_tgt = dates_tgt[idate]
        
        # Retrieve firing rate for each neuron
        neuronID_list = list(bhvevents_aligned_FRs_allevents_all_dates[date_tgt][act_animal_to_ana+' pull'].keys())
        nneurons = np.shape(neuronID_list)[0]
        
        # --- NEW: Storage for this session's multi-panel plot ---
        date_s_mean = None 
        date_neuron_traces = {}
        date_neuron_stats = {}
        
        for ineuron in np.arange(0,nneurons,1):
            
            neuronID = neuronID_list[ineuron]
            
            FR_ineuron = bhvevents_aligned_FRs_allevents_all_dates[date_tgt][act_animal_to_ana+' pull']\
                         [neuronID]['FR_allevents']

            # load and reconstruct the bhv weight
            try:
                all_betas_df = glm_datas_all_dates[date_tgt][(act_animal_to_ana, 'summary_df')]
                all_variables = pre_data_for_GLM_alldates[date_tgt][(act_animal_to_ana, 'X_all')]
                all_pulls = pre_data_for_GLM_alldates[date_tgt][(act_animal_to_ana, 'Y')]
            except:
                continue

            # 1. Prepare Design Matrix and Calculate Score
            X_scaled = scaler.transform(all_variables) 
            X_design = sm.add_constant(X_scaled, has_constant='add')
            weights = all_betas_df['Coefficient'].values

            full_session_score = np.dot(X_design, weights)
            all_session_scores[date_tgt] = full_session_score

            # 2. Extract Pull-Aligned Snapshots (-4s to 4s)
            pull_indices = np.where(all_pulls == 1)[0]
            window_frames = int(4 * fps)

            date_pull_matrix = []
            for idx in pull_indices:
                start = idx - window_frames
                end = idx + window_frames
                if start >= 0 and end <= len(full_session_score):
                    date_pull_matrix.append(full_session_score[start:end])

            score_matrix = np.array(date_pull_matrix)
            all_pull_snapshots[date_tgt] = score_matrix

            # 1. Align the pull numbers first
            min_pulls = min(score_matrix.shape[0], FR_ineuron.T.shape[0])

            s_mat = score_matrix[:min_pulls, :]
            f1 = FR_ineuron.T[:min_pulls, :]

            # Filter NaNs
            valid_mask = (~np.isnan(score_matrix).any(axis=1) & 
                          ~np.isnan(f1).any(axis=1))

            s_clean = score_matrix[valid_mask]
            f1_clean = f1[valid_mask]

            # 2. Calculate Correlations for the mean trace
            if 1: 
                s_mean = np.mean(s_clean, axis=0)
                f1_mean = np.mean(f1_clean, axis=0)
                
                r1, p1 = pearsonr(s_mean, f1_mean)

            # Store global stats for the swarm plot later
            all_p1.append(p1)
            all_r1.append(r1)
            
            # --- NEW: Store data for the session grid plot ---
            if date_s_mean is None:
                date_s_mean = s_mean # The GLM score trace is identical for all neurons on this date
                
            date_neuron_traces[neuronID] = f1_mean
            date_neuron_stats[neuronID] = {'r': r1, 'p': p1}

        # --- NEW: Generate 5-Column Grid Plot for the Session ---
        if nneurons > 0 and date_s_mean is not None:
            cols = 5
            rows = int(np.ceil(nneurons / cols))
            
            # Create a large figure based on the number of rows
            fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 2.5))
            axes = axes.flatten() if nneurons > 1 else [axes]
            
            time_axis = np.linspace(-4, 4, len(date_s_mean))
            
            for idx, (nID, f_mean) in enumerate(date_neuron_traces.items()):
                ax = axes[idx]
                stats = date_neuron_stats[nID]
                
                # Plot GLM Score on Primary Y-Axis (Black)
                ax.plot(time_axis, date_s_mean, color='black', lw=2, label='GLM')
                ax.set_ylabel('GLM', color='black', fontsize=8)
                ax.tick_params(axis='y', labelcolor='black', labelsize=7)
                
                # Plot Firing Rate on Secondary Y-Axis (Color-coded by significance)
                ax2 = ax.twinx()
                fr_color = 'red' if stats['p'] < 0.01 else 'gray'
                ax2.plot(time_axis, f_mean, color=fr_color, lw=2, alpha=0.8, label='FR')
                ax2.set_ylabel('FR (Hz)', color=fr_color, fontsize=8)
                ax2.tick_params(axis='y', labelcolor=fr_color, labelsize=7)
                
                # Panel Title with Stats
                ax.set_title(f"{nID}\nr={stats['r']:.2f}, p={stats['p']:.3f}", fontsize=9)
                ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
                
                # Only show x-label on the bottom row to keep it clean
                if idx >= len(axes) - cols: 
                    ax.set_xlabel('Time from pull (s)', fontsize=8)
                    
            # Turn off any empty subplots in the grid
            for idx in range(nneurons, len(axes)):
                axes[idx].axis('off')
                
            plt.suptitle(f'Neuron-Behavior Temporal Overlap: {date_tgt}\n(Red trace = p < 0.01)', fontsize=14)
            plt.tight_layout(rect=[0, 0, 1, 0.95]) # Leave room for suptitle
            plt.show()
        
    
    
    # --- Post-Loop Visualization ---
    # Create DataFrame for plotting
    corrs_df = pd.DataFrame({'neurons': all_r1})
    p_values_df = pd.DataFrame({'neurons': all_p1})

    # Melt data for seaborn
    plot_data = corrs_df.melt(var_name='neurons', value_name='corr coef')
    p_values_flat = p_values_df.melt(var_name='neurons', value_name='p')['p']

    # Define threshold and create a specific Hue column
    p_threshold = 0.01
    plot_data['Significance'] = ['Significant' if p < p_threshold else 'Not Significant' for p in p_values_flat]

    # Plotting
    plt.figure(figsize=(6, 5))

    # Swarm plot
    # We map 'hue' directly to our 'Significance' column
    sns.swarmplot(x='neurons', y='corr coef', data=plot_data, size=8, 
                  hue='Significance', 
                  palette={'Significant': 'red', 'Not Significant': 'gray'})
    

    # plt.ylabel(r'Variance Explained ($R^2$)')
    plt.ylabel(r'corr coef')
    plt.title(f'Predictive Power of Neural FRs across neurons \n(Red: p < {p_threshold} per neuron)')
    plt.grid(axis='y', alpha=0.3)

    # Move legend outside the plot so it doesn't cover data
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
        
        


In [ ]:
np.shape(all_r1)

In [ ]:
np.shape(FRPC1)